# N0 — Exploratory Data Analysis (Revised)
# Red-thread structure:
#   1. The Dataset — attrition funnel + industry composition
#   2. The Valuation Problem — staircase + within-industry dispersion
#   3. What Drives Valuation? — financial feature space
#   4. The Text Dimension — summarisation quality + semantic differentiation
#   5. Bridge → Methodology


In [ ]:
# imports & config
import sys
from pathlib import Path

notebook_dir = Path('__file__').parent if '__file__' in dir() else Path.cwd()
repo_root = next(
    (p for p in [notebook_dir, *notebook_dir.parents] if (p / 'config.py').exists()),
    None
)
if repo_root is None:
    raise FileNotFoundError('config.py not found — check repo structure')
sys.path.insert(0, str(repo_root))

from config import *
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
import json
import warnings
warnings.filterwarnings('ignore')

# ── Publication style ──────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor'  : 'white',
    'axes.facecolor'    : 'white',
    'axes.edgecolor'    : '#333333',
    'axes.linewidth'    : 0.8,
    'axes.grid'         : True,
    'grid.color'        : '#e5e5e5',
    'grid.linewidth'    : 0.6,
    'grid.alpha'        : 1.0,
    'xtick.direction'   : 'out',
    'ytick.direction'   : 'out',
    'xtick.major.size'  : 3,
    'ytick.major.size'  : 3,
    'font.family'       : 'serif',
    'font.size'         : 10,
    'axes.titlesize'    : 11,
    'axes.labelsize'    : 10,
    'legend.fontsize'   : 9,
    'legend.framealpha' : 0.9,
    'legend.edgecolor'  : '#cccccc',
    'figure.dpi'        : 150,
    'savefig.dpi'       : FIGURE_DPI,
    'savefig.bbox'      : 'tight',
    'savefig.facecolor' : 'white',
})

# Colour palette — accessible, print-friendly
C0    = '#2166ac'   # blue   — M0
C1    = '#d6604d'   # red    — M1
C2    = '#4dac26'   # green  — M2
C3    = '#8073ac'   # purple — M3
CGREY = '#888888'

print('Config loaded. Publication style set.')
print(f'  Figures saved to: {FIGURES}')


In [ ]:
# I/O declaration
INPUTS  = [PANEL_CLEAN, MULTIPLES, PEERS_M0, PEERS_M1, PEERS_M2, PEERS_M3, RESULTS_MAIN]
OUTPUTS = [FIGURES / f'eda_{x}.pdf' for x in [
    'attrition_funnel',
    'industry_composition',
    'valuation_staircase',
    'longitudinal',
    'size_and_features',
    'text_compression',
    'tfidf',
    'coverage_by_sector',
]]
# PURPOSE : EDA figures for thesis Section 3.4
# RUNTIME : ~8 min
# DEPENDS : panel_clean, multiples, peers_m0–m3, results_main,
#           business_summaries_{year}.csv


## 1 · The Dataset

In [ ]:
# ── Load data ─────────────────────────────────────────────────────────────────
df       = pd.read_parquet(PANEL_CLEAN)
mult     = pd.read_parquet(MULTIPLES)
peers_m0 = pd.read_parquet(PEERS_M0).dropna(subset=['focal_tic','peer_tic'])
peers_m1 = pd.read_parquet(PEERS_M1).dropna(subset=['focal_tic','peer_tic'])
peers_m2 = pd.read_parquet(PEERS_M2).dropna(subset=['focal_tic','peer_tic'])
peers_m3 = pd.read_parquet(PEERS_M3).dropna(subset=['focal_tic','peer_tic'])
results  = pd.read_csv(RESULTS_MAIN)
emb      = pd.read_parquet(EMBEDDINGS)

# ── Build evaluation sample from Gemini summary files ─────────────────────────
INVALID_FLAGS = {'ERROR', 'INSUFFICIENT_DATA', 'ERROR_EXTRACTING_TEXT'}
eval_tickers = set()
for yr in YEARS:
    path = SUMMARIES_FILES[yr]
    if not path.exists():
        print(f'  WARNING: {path} not found')
        continue
    df_s = pd.read_csv(path)
    valid = df_s[
        df_s['business_description'].notna() &
        ~df_s['business_description'].isin(INVALID_FLAGS) &
        (df_s['business_description'].str.len() > 50)
    ]
    for _, row in valid.iterrows():
        eval_tickers.add((row['tic'], int(row['fyear'])))

def restrict_to_eval(peers_df):
    mask = peers_df.apply(
        lambda r: (r['focal_tic'], int(r['focal_fyear'])) in eval_tickers, axis=1
    )
    return peers_df[mask].copy()

peers_m0 = restrict_to_eval(peers_m0)
peers_m1 = restrict_to_eval(peers_m1)
peers_m2 = restrict_to_eval(peers_m2)
peers_m3 = restrict_to_eval(peers_m3)
mult_eval = mult[mult.apply(
    lambda r: (r['tic'], int(r['fyear'])) in eval_tickers, axis=1
)].copy()

df_panel = df[['tic','fyear','ff49_num','industry','market_cap']].copy()
mult     = mult_eval  # default for all valuation figures

# ── Merge embeddings with industry labels (dispersion analysis) ───────────────
emb_cols = [c for c in emb.columns if c not in ['tic', 'fyear']]
emb_merged = emb.merge(
    df_panel[['tic','fyear','ff49_num','industry']].drop_duplicates(['tic','fyear']),
    on=['tic','fyear'], how='inner'
)

print(f'Full financial panel : {df.shape[0]:,} rows | {df["tic"].nunique():,} firms')

### 1a · Sample Attrition Funnel
How 170k raw observations become 13,559 evaluation firm-years.

In [ ]:
# ── Figure 1: Sample Attrition Funnel ────────────────────────────────────────
# Shows the reader exactly how we arrive at our dataset before anything else.
# Horizontal bar waterfall so the funnel shape is visually clear.

attrition = [
    ('Raw Compustat universe',                   169_724),
    ('After USD filter',                          141_563),
    ('After deduplication',                       127_948),
    ('After penny stocks & micro-caps (<$1/$50M)',  84_513),
    ('After quality filter (EV/SEQ/AT/SALE > 0)',   54_682),
    ('Restricted to fiscal years 2020–2024',        20_883),
    ('Evaluation sample (valid 10-K summary)',       13_559),
]

labels   = [a[0] for a in attrition]
values   = [a[1] for a in attrition]
retained = [v / attrition[0][1] * 100 for v in values]

fig, ax = plt.subplots(figsize=(9, 4.2))

bar_colors = [C0] * (len(attrition) - 1) + ['#1a6b1a']   # final bar green
bars = ax.barh(range(len(attrition)), values,
               color=bar_colors, height=0.58,
               edgecolor='white', linewidth=0.5)

# Percentage labels inside bars
for i, (bar, val, pct) in enumerate(zip(bars, values, retained)):
    xpos = val / 2
    ax.text(xpos, bar.get_y() + bar.get_height() / 2,
            f'{val:,.0f}  ({pct:.0f}%)',
            ha='center', va='center',
            color='white', fontsize=8.5, fontweight='bold')

ax.set_yticks(range(len(attrition)))
ax.set_yticklabels(labels, fontsize=9)
ax.set_xlabel('Firm-year observations', fontsize=9)
ax.set_xlim(0, attrition[0][1] * 1.05)
ax.invert_yaxis()   # top = raw, bottom = eval sample
ax.xaxis.set_major_formatter(
    plt.FuncFormatter(lambda x, _: f'{x/1000:.0f}k')
)
ax.set_title('From 170k to 13.6k: How We Arrive at the Evaluation Sample',
             fontsize=10, pad=10)

# Vertical reference line at eval sample
ax.axvline(13_559, color='#1a6b1a', linewidth=1.2,
           linestyle='--', alpha=0.6)

plt.tight_layout()
plt.savefig(FIGURES / 'eda_attrition_funnel.pdf')
plt.show()
print(f"Saved: {FIGURES / 'eda_attrition_funnel.pdf'}")


### 1b · Industry Composition
Who remains after filtering — and does filtering distort the mix?

In [ ]:
# ── Figure 2: Industry Composition — Full Panel vs Evaluation Sample ──────────
# Groups FF49 into 7 readable macro-sectors.
# Side-by-side bars show what filtering keeps vs. removes.

sector_map = {
    'Technology & Software':
        ['Computer Software','Computers','Electronic Equipment',
         'Measuring and Control Eq.','Communication'],
    'Business & Financial Services':
        ['Business Services','Banking','Financial Trading',
         'Insurance','Real Estate'],
    'Healthcare & Pharma':
        ['Pharmaceutical Products','Medical Equipment','Healthcare'],
    'Energy & Industrials':
        ['Petroleum and Natural Gas','Chemicals','Machinery',
         'Electrical Equipment','Steel Works Etc','Construction',
         'Construction Materials','Defense','Mining'],
    'Consumer':
        ['Retail','Food Products','Consumer Goods','Apparel',
         'Restaurants, Hotels, Motels','Recreation',
         'Beer & Liquor','Tobacco Products','Candy & Soda'],
    'Other':
        ['Automobiles and Trucks','Rubber and Plastic Products',
         'Fabricated Products','Shipbuilding, Railroad Eq.',
         'Business Supplies','Printing and Publishing',
         'Shipping Containers','Textiles','Agriculture',
         'Transportation','Personal Services','Utilities',
         'Aircraft','Entertainment','Precious Metals',
         'Non-Metallic Mining','Coal','Almost Nothing'],
}
def assign_sector(ind):
    for sector, inds in sector_map.items():
        if ind in inds:
            return sector
    return 'Other'

df['sector']        = df['industry'].map(assign_sector)
df_eval             = df[df.apply(lambda r: (r['tic'], r['fyear']) in eval_tickers, axis=1)]

full_counts = df['sector'].value_counts()
eval_counts = df_eval['sector'].value_counts()
sectors     = list(sector_map.keys())
full_vals   = [full_counts.get(s, 0) for s in sectors]
eval_vals   = [eval_counts.get(s, 0) for s in sectors]

x     = np.arange(len(sectors))
width = 0.38

fig, ax = plt.subplots(figsize=(10, 4.5))
b1 = ax.bar(x - width/2, full_vals, width, label='Full panel (20,883)',
            color=C0, alpha=0.85, edgecolor='white', linewidth=0.4)
b2 = ax.bar(x + width/2, eval_vals, width, label='Evaluation sample (13,559)',
            color=C2, alpha=0.85, edgecolor='white', linewidth=0.4)

# Value labels
for bar in list(b1) + list(b2):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 30,
            f'{h:,.0f}', ha='center', va='bottom', fontsize=7.2)

ax.set_xticks(x)
# Clean short sector labels for x-axis
short_labels = {
    'Technology & Software':          'Tech &\nSoftware',
    'Business & Financial Services':  'Business &\nFinancial',
    'Healthcare & Pharma':            'Healthcare\n& Pharma',
    'Energy & Industrials':           'Energy &\nIndustrials',
    'Consumer':                       'Consumer',
    'Other':                          'Other',
}
ax.set_xticklabels(
    [short_labels.get(s, s) for s in sectors],
    fontsize=9, rotation=0, ha='center'
)
ax.set_ylabel('Firm-year observations', fontsize=9)
ax.set_title('Industry Composition: Full Panel vs. Evaluation Sample',
             fontsize=10, pad=10)
ax.legend(fontsize=8.5)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))

plt.tight_layout()
plt.savefig(FIGURES / 'eda_industry_composition.pdf')
plt.show()
print(f"Saved: {FIGURES / 'eda_industry_composition.pdf'}")


In [ ]:
# ── FF49 Industry Detail Table ────────────────────────────────────────────────
# Firm-year counts per FF49 industry: full panel vs eval sample

INNOVATION_FF49 = [
    "Computer Software", "Computers", "Electronic Equipment",
    "Measuring and Control Eq.", "Business Services",
    "Pharmaceutical Products", "Medical Equipment", "Communication",
]
TRADITIONAL_FF49 = [
    "Food Products", "Beer & Liquor", "Tobacco Products",
    "Textiles", "Apparel", "Business Supplies", "Printing and Publishing",
    "Wholesale", "Retail", "Restaurants, Hotels, Motels",
    "Construction", "Steel Works Etc", "Non-Metallic Mining",
]

def stratum(ind):
    if ind in INNOVATION_FF49:  return "Innovation"
    if ind in TRADITIONAL_FF49: return "Traditional"
    return "Other"

# Full panel counts
full_counts = df.groupby('industry').agg(
    full_fy=('tic', 'count'),
    full_firms=('tic', 'nunique')
).reset_index()

# Eval sample counts
df_eval_ind = df[df.apply(lambda r: (r['tic'], r['fyear']) in eval_tickers, axis=1)]
eval_counts = df_eval_ind.groupby('industry').agg(
    eval_fy=('tic', 'count'),
    eval_firms=('tic', 'nunique')
).reset_index()

summary = full_counts.merge(eval_counts, on='industry', how='left').fillna(0)
summary['eval_fy']    = summary['eval_fy'].astype(int)
summary['eval_firms'] = summary['eval_firms'].astype(int)
summary['retention']  = (summary['eval_fy'] / summary['full_fy'] * 100).round(1)
summary['stratum']    = summary['industry'].map(stratum)

# Sort: stratum then by eval firm-years descending
order = {"Innovation": 0, "Traditional": 1, "Other": 2}
summary['_ord'] = summary['stratum'].map(order)
summary = summary.sort_values(['_ord', 'eval_fy'], ascending=[True, False]).drop(columns='_ord')

# Display
pd.set_option('display.max_rows', 60)
pd.set_option('display.width', 120)
print(f"{'Industry':<35} {'Stratum':<14} {'Full FY':>7} {'Full Firms':>10} "
      f"{'Eval FY':>7} {'Eval Firms':>10} {'Ret.%':>6}")
print("-" * 100)
for _, r in summary.iterrows():
    flag = " ⚠" if r['eval_fy'] < 100 else ""
    print(f"{r['industry']:<35} {r['stratum']:<14} {r['full_fy']:>7,} {r['full_firms']:>10,} "
          f"{r['eval_fy']:>7,} {r['eval_firms']:>10,} {r['retention']:>5.1f}%{flag}")

print()
print(f"Innovation total: {summary[summary['stratum']=='Innovation']['eval_fy'].sum():,} firm-years")
print(f"Traditional total: {summary[summary['stratum']=='Traditional']['eval_fy'].sum():,} firm-years")
print(f"Other total: {summary[summary['stratum']=='Other']['eval_fy'].sum():,} firm-years")


## 2 · The Valuation Problem

### 2a · The Staircase
Industry explains *something* — but within-industry variance is the real story.

In [ ]:
# ── Figure 3: Valuation Staircase ─────────────────────────────────────────────
# The core problem: within-industry dispersion is huge.
# Compact: 7-inch height, industry names readable at 8pt.

industry_stats = mult.groupby('industry')['ln_v2s'].agg(
    median='median', q25=lambda x: x.quantile(0.25),
    q75=lambda x: x.quantile(0.75), n='count'
).reset_index().sort_values('median')

groups   = [mult[mult['industry'] == ind]['ln_v2s'].dropna().values
            for ind in industry_stats['industry']]
f_stat, p_val = stats.f_oneway(*groups)

fig, ax = plt.subplots(figsize=(8, 9))

cmap   = plt.cm.RdYlGn(np.linspace(0.12, 0.88, len(industry_stats)))
y_pos  = range(len(industry_stats))

# IQR bars (light background)
ax.barh(list(y_pos),
        industry_stats['q75'].values - industry_stats['q25'].values,
        left=industry_stats['q25'].values,
        color=cmap, height=0.65, alpha=0.35,
        edgecolor='none', label='IQR')

# Median dots
ax.scatter(industry_stats['median'].values, list(y_pos),
           color=cmap, s=28, zorder=4, edgecolors='white', linewidths=0.4)

overall_med = mult['ln_v2s'].median()
ax.axvline(overall_med, color=CGREY, linewidth=1, linestyle='--',
           label=f'Overall median ({overall_med:.2f})', zorder=3)
ax.axvline(0, color='#333333', linewidth=0.6, linestyle='-', zorder=3)

ax.set_yticks(list(y_pos))
ax.set_yticklabels(industry_stats['industry'].values, fontsize=7.8)
ax.set_xlabel('ln(EV / Sales)', fontsize=9)
ax.set_title(
    f'Valuation Staircase across FF49 Industries\n'
    f'(ANOVA F = {f_stat:.1f}, p < 0.001, n = {len(mult):,} firm-years)',
    fontsize=10, pad=10
)
ax.legend(fontsize=8, loc='lower right')

plt.tight_layout()
plt.savefig(FIGURES / 'eda_valuation_staircase.pdf')
plt.show()
print(f"Saved: {FIGURES / 'eda_valuation_staircase.pdf'}")
print(f"ANOVA: F={f_stat:.2f}, p={p_val:.2e}")


In [ ]:
# ── Validate IQR numbers cited in thesis text ─────────────────────────────────
for industry in ["Pharmaceutical Products", "Computer Software", "Trading",
                 "Banking", "Business Services"]:
    d = mult[mult["industry"] == industry]["ln_v2s"].dropna()
    q25, q75 = d.quantile(0.25), d.quantile(0.75)
    iqr       = q75 - q25
    ratio     = np.exp(iqr)   # back-transform: factor difference between Q75 and Q25
    print(f"{industry:<30}  Q25={q25:.3f}  Q75={q75:.3f}  "
          f"IQR={iqr:.3f} log-units  → {ratio:.1f}x multiple spread  n={len(d)}")

### 2b · Temporal Stability
The fundamental-to-multiple mapping shifts; annual cross-sections are essential.

In [ ]:
# ── Figure 4: Longitudinal Trends (2020–2024) ─────────────────────────────────
# Motivates annual cross-sectional construction (not pooling).
# Clean dual-axis, narrower to pair with Figure 3 in the thesis layout.

yearly  = mult.groupby('fyear')['ln_v2s'].agg(['median','std']).reset_index()
roa_yr  = df.groupby('fyear')['roa_wr'].median().reset_index()
roa_yr.columns = ['fyear', 'median_roa']

fig, ax1 = plt.subplots(figsize=(6, 3.8))

ax1.plot(yearly['fyear'], yearly['median'] * 100,
         color=C0, marker='o', linewidth=2, markersize=5,
         label='Median ln(EV/Sales) × 100')
ax1.fill_between(
    yearly['fyear'],
    (yearly['median'] - yearly['std']) * 100,
    (yearly['median'] + yearly['std']) * 100,
    alpha=0.10, color=C0
)
ax1.set_xlabel('Fiscal Year', fontsize=9)
ax1.set_ylabel('Median ln(EV/Sales) × 100', color=C0, fontsize=9)
ax1.tick_params(axis='y', labelcolor=C0)
ax1.set_xticks(YEARS)

ax2 = ax1.twinx()
ax2.plot(roa_yr['fyear'], roa_yr['median_roa'] * 100,
         color=C1, marker='s', linewidth=2, markersize=5,
         linestyle='--', label='Median ROA (%)')
ax2.set_ylabel('Median ROA (%)', color=C1, fontsize=9)
ax2.tick_params(axis='y', labelcolor=C1)

lines1, lab1 = ax1.get_legend_handles_labels()
lines2, lab2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, lab1 + lab2, fontsize=8, loc='upper right')

ax1.set_title('Median Valuation and Profitability by Fiscal Year (2020–2024)',
              fontsize=10, pad=8)
plt.tight_layout()
plt.savefig(FIGURES / 'eda_longitudinal.pdf')
plt.show()
print(f"Saved: {FIGURES / 'eda_longitudinal.pdf'}")


## 3 · What Drives Valuation?
Size is weak; financial ratios — especially asset turnover — are strong.

In [ ]:
# ── Figure 5: Two-panel — Size Effect + Top Financial Features ────────────────
# Left:  hexbin size vs valuation  → size alone is weak
# Right: top-15 Spearman features  → what actually matters
# Both panels reinforce the same point from different angles.

with open(SELECTED_FEATURES_FILE) as _f:
    selected_features = json.load(_f)['selected_features']

# ── data prep ────
df_size = df[['tic','fyear','market_cap']].copy()
df_size['ln_mktcap'] = np.log(df_size['market_cap'].clip(lower=1))
df_size = df_size.merge(mult[['tic','fyear','ln_v2s']], on=['tic','fyear'])
df_size = df_size.dropna(subset=['ln_mktcap','ln_v2s'])

df_corr = df[['tic','fyear'] + selected_features].merge(
    mult[['tic','fyear','ln_v2s']], on=['tic','fyear'], how='inner'
)
signed_corr = (df_corr[selected_features + ['ln_v2s']]
               .corr(method='spearman')['ln_v2s']
               .drop('ln_v2s'))
top15 = signed_corr.abs().sort_values(ascending=False).head(15)
top15_signed = signed_corr.reindex(top15.index)

m, b, r, p, _ = stats.linregress(df_size['ln_mktcap'], df_size['ln_v2s'])

# ── plot ────
fig, (ax_left, ax_right) = plt.subplots(1, 2, figsize=(12, 5))

# Left: hexbin
hb = ax_left.hexbin(df_size['ln_mktcap'], df_size['ln_v2s'],
                    gridsize=48, cmap='Blues', mincnt=1, linewidths=0.08)
fig.colorbar(hb, ax=ax_left, label='Count', shrink=0.82)
x_range = np.linspace(df_size['ln_mktcap'].quantile(0.01),
                      df_size['ln_mktcap'].quantile(0.99), 100)
ax_left.plot(x_range, m * x_range + b, color=C1, linewidth=1.6,
             label=f'OLS (β={m:.3f}, R²={r**2:.3f})')
ax_left.set_xlabel('ln(Market Cap, USD m)', fontsize=9)
ax_left.set_ylabel('ln(EV / Sales)', fontsize=9)
ax_left.set_title('Size vs. Valuation Multiple', fontsize=10)
ax_left.legend(fontsize=8)

# Right: horizontal bar, colour by sign
bar_cols = [C0 if v > 0 else C1 for v in top15_signed.values]
y_pos    = range(len(top15))
ax_right.barh(list(y_pos), top15.values,
              color=bar_cols, height=0.65,
              edgecolor='white', linewidth=0.3)
ax_right.set_yticks(list(y_pos))
ax_right.set_yticklabels(top15.index, fontsize=8.2)
ax_right.set_xlabel('|Spearman ρ| with ln(EV/Sales)', fontsize=9)
ax_right.set_title('Top 15 Financial Predictors of Valuation', fontsize=10)
ax_right.axvline(0.1, color=CGREY, linewidth=0.8, linestyle=':')
for i, (feat, val) in enumerate(zip(top15.index, top15.values)):
    ax_right.text(val + 0.004, i, f'{val:.3f}', va='center', fontsize=7.4)

# Legend for sign
from matplotlib.patches import Patch
ax_right.legend(
    handles=[Patch(facecolor=C0, label='Positive association'),
             Patch(facecolor=C1, label='Negative association')],
    fontsize=7.5, loc='lower right'
)

plt.suptitle(
    'Financial Predictors of Valuation: Size Explains <2%, Ratios Explain Much More',
    fontsize=10, y=1.01
)
plt.tight_layout()
plt.savefig(FIGURES / 'eda_size_and_features.pdf')
plt.show()
print(f"Saved: {FIGURES / 'eda_size_and_features.pdf'}")
print(f"Top feature: {top15.index[0]}  ρ = {top15.values[0]:.3f}")
print(f"OLS size-valuation: β={m:.3f}, R²={r**2:.3f}")


## 4 · The Text Dimension
Gemini summaries are consistent, dense, and semantically differentiated across industries.

In [ ]:
# ── Figure 6: Text Pipeline Quality ──────────────────────────────────────────
# 2-panel: left = word count histograms per year (overlaid KDE, compact)
#          right = TF-IDF signatures for 6 industries (2×3 grid)
# Shown as two separate saved figures so they can be placed independently in LaTeX.

INVALID_FLAGS = {'ERROR', 'INSUFFICIENT_DATA', 'ERROR_EXTRACTING_TEXT'}
frames = []
for yr in YEARS:
    path = SUMMARIES_FILES[yr]
    if not path.exists():
        continue
    df_s = pd.read_csv(path)
    df_s = df_s[
        df_s['business_description'].notna() &
        ~df_s['business_description'].isin(INVALID_FLAGS) &
        (df_s['business_description'].str.len() > 50)
    ].copy()
    df_s['word_count'] = df_s['business_description'].apply(lambda x: len(str(x).split()))
    df_s['fyear'] = yr
    frames.append(df_s[['tic','fyear','business_description','word_count']])

df_text = pd.concat(frames, ignore_index=True)
df_text = df_text.merge(
    df_panel[['tic','fyear','industry']].drop_duplicates(),
    on=['tic','fyear'], how='left'
)
print(f"Text corpus: {len(df_text):,} firm-years, {df_text['industry'].nunique()} industries")

# ── Figure 6a: word count distribution (compact 5-panel) ─────────────────────
fig, axes = plt.subplots(1, 5, figsize=(12, 3.2), sharey=True)
for ax, yr in zip(axes, YEARS):
    data = df_text[df_text['fyear'] == yr]['word_count']
    ax.hist(data, bins=40, color=C0, alpha=0.80, edgecolor='none')
    ax.axvline(400,          color=C1,    linewidth=1.5, linestyle='--', label='Target 400w')
    ax.axvline(data.median(), color=CGREY, linewidth=1.2, linestyle=':',
               label=f'Med. {data.median():.0f}w')
    ax.set_title(f'FY {yr}\n(n={len(data):,})', fontsize=8.5)
    ax.set_xlabel('Word count', fontsize=8)
    if ax == axes[0]:
        ax.set_ylabel('Frequency', fontsize=8)
    ax.legend(fontsize=6.5, handlelength=1.2)
    ax.tick_params(labelsize=7.5)

plt.suptitle('Gemini Summary Word Count Distribution by Fiscal Year', fontsize=10, y=1.02)
plt.tight_layout()
plt.savefig(FIGURES / 'eda_text_compression.pdf')
plt.show()
print(f"Saved: {FIGURES / 'eda_text_compression.pdf'}")

# ── Figure 6b: TF-IDF industry signatures ────────────────────────────────────
from sklearn.feature_extraction.text import TfidfVectorizer

FOCUS = ['Computer Software','Pharmaceutical Products',
         'Banking','Retail','Petroleum and Natural Gas','Medical Equipment']

df_focus = df_text[df_text['industry'].isin(FOCUS)].copy()
industry_docs = {
    ind: ' '.join(df_focus[df_focus['industry'] == ind]['business_description'].fillna(''))
    for ind in FOCUS
}

vec = TfidfVectorizer(stop_words='english', max_features=5000,
                      ngram_range=(1,2), max_df=0.85, min_df=2)
tfidf_mat    = vec.fit_transform(list(industry_docs.values()))
feature_names = vec.get_feature_names_out()

fig, axes = plt.subplots(2, 3, figsize=(11, 6.5))
axes = axes.flatten()

for ax, (ind, idx) in zip(axes, zip(FOCUS, range(len(FOCUS)))):
    scores    = tfidf_mat[idx].toarray().flatten()
    top_idx   = scores.argsort()[-10:][::-1]
    top_terms  = [feature_names[i] for i in top_idx]
    top_scores = [scores[i] for i in top_idx]
    ax.barh(top_terms[::-1], top_scores[::-1],
            color=C0, alpha=0.82, edgecolor='none')
    ax.set_title(ind, fontsize=8.8, fontweight='bold')
    ax.set_xlabel('TF-IDF score', fontsize=7.5)
    ax.tick_params(labelsize=7.5)

plt.suptitle('Top TF-IDF Terms by FF49 Industry (Gemini Summary Corpus)',
             fontsize=10, y=1.01)
plt.tight_layout()
plt.savefig(FIGURES / 'eda_tfidf.pdf')
plt.show()
print(f"Saved: {FIGURES / 'eda_tfidf.pdf'}")


In [ ]:
# Coverage rate by macro-sector
sector_map = {
    'Technology & Software': [
        'Computer Software', 'Computers', 'Electronic Equipment',
        'Measuring and Control Eq.', 'Communication'
    ],
    'Business & Financial Services': [
        'Business Services', 'Banking', 'Insurance',
        'Trading', 'Real Estate', 'Personal Services'
    ],
    'Healthcare & Pharma': [
        'Pharmaceutical Products', 'Medical Equipment', 'Healthcare'
    ],
    'Energy & Industrials': [
        'Petroleum and Natural Gas', 'Utilities', 'Construction',
        'Steel Works Etc', 'Machinery', 'Electrical Equipment',
        'Chemicals', 'Construction Materials'
    ],
    'Consumer': [
        'Retail', 'Wholesale', 'Food Products',
        'Restaurants, Hotels, Motels', 'Apparel', 'Consumer Goods'
    ],
    'Other Manufacturing': [
        'Automobiles and Trucks', 'Aircraft', 'Defense',
        'Fabricated Products', 'Rubber and Plastic Products',
        'Business Supplies', 'Printing and Publishing'
    ],
}

# Build reverse map
ind_to_sector = {}
for sector, industries in sector_map.items():
    for ind in industries:
        ind_to_sector[ind] = sector

df_panel['macro_sector'] = df_panel['industry'].map(ind_to_sector).fillna('Other')
eval_set = set(zip(peers_m3['focal_tic'], peers_m3['focal_fyear']))
df_panel['in_eval'] = df_panel.apply(
    lambda r: (r['tic'], r['fyear']) in eval_set, axis=1
)

coverage = (df_panel.groupby('macro_sector')
            .agg(total=('tic','count'), in_eval=('in_eval','sum'))
            .assign(coverage=lambda x: x['in_eval'] / x['total'] * 100)
            .sort_values('coverage', ascending=True))

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(coverage.index, coverage['coverage'],
               color='#4C72B0', alpha=0.85)
ax.axvline(coverage['coverage'].mean(), color='red',
           linewidth=1, linestyle='--',
           label=f"Mean {coverage['coverage'].mean():.1f}%")
for bar, val in zip(bars, coverage['coverage']):
    ax.text(val + 0.3, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=9)
ax.set_xlabel('Textual coverage rate (%)')
ax.set_title('Textual Coverage Rate by Macro-Sector', fontsize=11)
ax.legend()
ax.set_xlim(0, 85)
plt.tight_layout()
plt.savefig(FIGURES / 'eda_coverage_by_sector.pdf',
            dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()

## 5 · Bridge to Methodology

The EDA establishes four empirical facts that directly motivate the four-model framework:

1. **Industry alone is insufficient.** Within-industry IQR in ln(EV/Sales) spans 2+ log-units even after
   restricting to the evaluation sample — there is substantial peer-selection headroom above M0.
2. **Financial ratios carry signal.** Asset turnover (ρ = 0.576) and cash-conversion metrics explain
   far more cross-sectional valuation dispersion than firm size (R² < 2%), motivating M1.
3. **Text is semantically differentiated.** TF-IDF signatures are distinct across industries and
   summary lengths are consistent, giving FinBERT a high-quality, standardised input — motivating M2.
4. **The 2021–2022 decoupling.** Valuations and fundamentals diverge sharply during rate normalisation,
   confirming that the 2020–2024 window is a demanding test bed for all models.

These four facts set up a natural question: **can we do better by combining both signals?** That is
precisely what M3 — late fusion of financial similarity and textual similarity — is designed to test.


In [ ]:
# ── Descriptive Statistics Table — Financial Panel vs. Evaluation Sample ──────
# Computes median and mean for key size and valuation variables.
# Financial panel uses panel_clean directly.
# Evaluation sample multiples come from mult (already restricted to eval_tickers).
# Raw multiples (ev_sales etc.) are winsorised at 1/99 per fyear — matches thesis.

# ── Financial panel subset ─────────────────────────────────────────────────────
panel_vars = df[['tic', 'fyear', 'market_cap', 'ev', 'at', 'seq']].copy()
panel_vars['ev_sales']       = df['ev'] / df['sale']
panel_vars['ev_assets']      = df['ev'] / df['at']
panel_vars['mktcap_book']    = df['market_cap'] / df['seq']

# Winsorise raw multiples at 1/99 per fyear to match multiples.parquet treatment
def winsorise(s, lo=0.01, hi=0.99):
    low, high = s.quantile(lo), s.quantile(hi)
    return s.clip(low, high)

for col in ['ev_sales', 'ev_assets', 'mktcap_book']:
    panel_vars[col] = panel_vars.groupby('fyear')[col].transform(winsorise)

# ── Evaluation sample subset ───────────────────────────────────────────────────
eval_mask   = panel_vars.apply(
    lambda r: (r['tic'], int(r['fyear'])) in eval_tickers, axis=1
)
eval_vars   = panel_vars[eval_mask].copy()

# ── Compute stats ──────────────────────────────────────────────────────────────
metrics = {
    'Market Cap ($M)' : 'market_cap',
    'Enterprise Value ($M)' : 'ev',
    'Total Assets ($M)' : 'at',
    'EV / Sales' : 'ev_sales',
    'EV / Assets' : 'ev_assets',
    'MktCap / Book Equity' : 'mktcap_book',
}

rows = []
for label, col in metrics.items():
    p_med  = panel_vars[col].dropna().median()
    p_mean = panel_vars[col].dropna().mean()
    e_med  = eval_vars[col].dropna().median()
    e_mean = eval_vars[col].dropna().mean()
    rows.append((label, p_med, p_mean, e_med, e_mean))

stats_df = pd.DataFrame(rows, columns=[
    'Variable', 'Panel Median', 'Panel Mean',
    'Eval Median', 'Eval Mean'
])

# ── Print for thesis table ─────────────────────────────────────────────────────
print("=" * 75)
print("SUMMARY STATISTICS — Financial Panel vs. Evaluation Sample")
print("=" * 75)
print(f"{'Variable':<25} {'Panel Med':>12} {'Panel Mean':>12} "
      f"{'Eval Med':>12} {'Eval Mean':>12}")
print("-" * 75)
for _, row in stats_df.iterrows():
    v = row['Variable']
    # Dollar variables in millions, multiples as-is
    if '$M' in v:
        fmt = lambda x: f"${x:>10,.0f}"
    else:
        fmt = lambda x: f"{x:>12.2f}"
    print(f"{v:<25} {fmt(row['Panel Median'])} {fmt(row['Panel Mean'])} "
          f"{fmt(row['Eval Median'])} {fmt(row['Eval Mean'])}")
print("-" * 75)
print(f"{'Firm-years':<25} {'20,883':>12} {'':>12} {'13,559':>12}")
print(f"{'Unique firms':<25} {'5,700':>12} {'':>12} {'3,494':>12}")
print("=" * 75)


In [ ]:
# ── Fiscal Year-End Distribution ───────────────────────────────────────────────
if 'datadate' in df.columns:
    df['fye_month'] = pd.to_datetime(df['datadate']).dt.month
else:
    # fallback if datadate not in panel_clean
    print("datadate not found in panel_clean — load from raw if needed")

fye_full = df['fye_month'].value_counts().sort_index()
fye_eval = df[eval_mask]['fye_month'].value_counts().sort_index()

month_names = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
               7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}

print("Fiscal Year-End Distribution")
print(f"{'Month':<6} {'Full FY':>10} {'Full %':>8} {'Eval FY':>10} {'Eval %':>8}")
print("-" * 48)
for m in range(1, 13):
    fn = fye_full.get(m, 0)
    en = fye_eval.get(m, 0)
    fp = fn / len(df) * 100
    ep = en / len(eval_mask[eval_mask].index) * 100
    print(f"{month_names[m]:<6} {fn:>10,} {fp:>7.1f}% {en:>10,} {ep:>7.1f}%")
print("-" * 48)
dec_pct_full = fye_full.get(12, 0) / len(df) * 100
dec_pct_eval = fye_eval.get(12, 0) / sum(eval_mask) * 100
print(f"\nDecember share: {dec_pct_full:.1f}% (full panel) | "
      f"{dec_pct_eval:.1f}% (eval sample)")